# 04 - Growth Analysis

## 1. Purpose
Compute the full growth-metric suite (MoM, YoY, running revenue, rolling
volatility, acceleration, and decline streaks) in Python, as an independent
implementation to reconcile against the SQL layer in `sql/03_analysis/` and
`sql/04_business_cases/`.


## 2. Imports

In [1]:
import pandas as pd
import numpy as np


## 3. Data Loading

In [1]:
df = pd.read_csv("../data/raw/monthly_revenue_raw.csv")
df['sales_month'] = pd.to_datetime(df['sales_month'], format='%Y-%m')
df = df.sort_values('sales_month').reset_index(drop=True)
df.rename(columns={'current_month_revenue': 'revenue'}, inplace=True)
df.head()


  sales_month  revenue
0  2016-10-01    43000
1  2016-11-01   105000
2  2016-12-01   253000
3  2017-01-01   390000
4  2017-02-01   362000

## 4. Analysis

In [2]:
# MoM growth -- first month is NULL by design, not 0
df['previous_month_revenue'] = df['revenue'].shift(1)
df['mom_growth_pct'] = (
    (df['revenue'] - df['previous_month_revenue']) / df['previous_month_revenue'] * 100
).round(2)

# Running revenue
df['running_revenue'] = df['revenue'].cumsum()

# YoY growth -- only valid from month 13 onward
df['revenue_12mo_ago'] = df['revenue'].shift(12)
df['yoy_growth_pct'] = (
    (df['revenue'] - df['revenue_12mo_ago']) / df['revenue_12mo_ago'] * 100
).round(2)

# Rolling volatility
df['rolling_3m_avg'] = df['revenue'].rolling(3).mean().round(2)
df['rolling_3m_std'] = df['revenue'].rolling(3).std().round(2)

# Growth acceleration: change in the MoM growth rate itself
df['revenue_acceleration'] = (df['mom_growth_pct'] - df['mom_growth_pct'].shift(1)).round(2)

df[['sales_month', 'revenue', 'mom_growth_pct', 'yoy_growth_pct', 'running_revenue']].tail(10)


   sales_month  revenue  mom_growth_pct  yoy_growth_pct  running_revenue
13  2017-11-01  1027013           61.73          878.11          6958013
14  2017-12-01   780000          -24.05          208.30          7738013
15  2018-01-01   952000           22.05          144.10          8690013
16  2018-02-01   861000           -9.56          137.85          9551013
17  2018-03-01  1015000           17.89           89.72         10566013
18  2018-04-01  1042000            2.66          130.02         11608013
19  2018-05-01  1061000            1.82          101.71         12669013
20  2018-06-01  1033000           -2.64           70.18         13702013
21  2018-07-01  1038514            0.53           63.55         14740527
22  2018-08-01   996974           -4.00           46.61         15737501

In [3]:
# Consecutive-decline streak detection (gaps-and-islands)
is_decline = (df['mom_growth_pct'] < 0).astype(int)
streak_group = (np.arange(len(df)) + 1) - is_decline.cumsum()
df['decline_streak'] = np.where(
    is_decline == 1,
    is_decline.groupby(streak_group).cumsum(),
    0
)
df[['sales_month', 'mom_growth_pct', 'decline_streak']].tail(8)


   sales_month  mom_growth_pct  decline_streak
15  2018-01-01           22.05               0
16  2018-02-01           -9.56               1
17  2018-03-01           17.89               0
18  2018-04-01            2.66               0
19  2018-05-01            1.82               0
20  2018-06-01           -2.64               1
21  2018-07-01            0.53               0
22  2018-08-01           -4.00               1

In [3]:
# Best / worst months
best_mom = df.loc[df['mom_growth_pct'].idxmax()]
worst_mom = df.loc[df['mom_growth_pct'].idxmin()]
highest_rev = df.loc[df['revenue'].idxmax()]
lowest_rev = df.loc[df['revenue'].idxmin()]

print("Best MoM growth month:", best_mom['sales_month'].date(), f"({best_mom['mom_growth_pct']}%)")
print("Worst MoM growth month:", worst_mom['sales_month'].date(), f"({worst_mom['mom_growth_pct']}%)")
print("Highest revenue month:", highest_rev['sales_month'].date(), f"({highest_rev['revenue']:,.0f})")
print("Lowest revenue month:", lowest_rev['sales_month'].date(), f"({lowest_rev['revenue']:,.0f})")


Best MoM growth month: 2016-11-01 (144.19%)
Worst MoM growth month: 2017-12-01 (-24.05%)
Highest revenue month: 2018-05-01 (1,061,000)
Lowest revenue month: 2016-10-01 (43,000)


In [3]:
# Recent performance: last 3 months vs prior 3 months
last3 = df.tail(3)['revenue'].mean()
prior3 = df.iloc[-6:-3]['revenue'].mean()
pct_change = (last3 - prior3) / prior3 * 100

print(f"Last 3 months avg revenue:  {last3:,.2f}")
print(f"Prior 3 months avg revenue: {prior3:,.2f}")
print(f"Change: {pct_change:.2f}%")


Last 3 months avg revenue:  1,022,829.33
Prior 3 months avg revenue: 1,039,333.33
Change: -1.59%


## 5. Findings

- **Reconciliation confirms the SQL layer**: every mom_growth_pct,
  running_revenue, and yoy_growth_pct value computed here matches
  `sql/03_analysis/02_mom_growth.sql`, `04_running_revenue.sql`, and
  `03_yoy_growth.sql` exactly (validated against the same source CSV via a
  SQLite cross-check during development).
- **Best MoM growth**: Nov-2016 at +144.19% -- driven by the near-zero
  October base, not a repeatable event. Interpret with caution (see
  `docs/metric_definitions.md`, "early-period ramp" caveat).
- **Worst MoM growth**: Dec-2017 at -24.05%, immediately following the
  Nov-2017 spike -- consistent with a one-off peak followed by reversion,
  not a sustained decline (Jan-2018 recovered to +22.05%).
- **Highest revenue month**: May-2018 (1,061,000). **Lowest**: Oct-2016
  (43,000, the first month of the series).
- **Decline streaks are short**: the longest run of consecutive
  MoM-negative months in this dataset is 1 -- there is **no 2+ month
  consecutive decline** anywhere in the 23-month series, including the
  most recent month (Aug-2018, -4.00%), which follows a positive July
  (+0.53%). This directly corrects an earlier, unverified claim in this
  project's original README that described the ending months as a
  "2-month consecutive decline" -- that claim does not hold up against
  the actual streak calculation and has been dropped.
- **Recent performance**: the last 3 months (Jun-Aug 2018) averaged
  1,022,829 vs. 1,039,333 for the prior 3 months (Mar-May 2018) -- a
  **-1.59% change**, essentially flat, not a meaningful deterioration.


## 6. Conclusion

The growth metrics are fully reconciled between SQL and Python. The
business narrative is: strong early ramp, a Nov-2017 one-off-looking
peak, reversion, and then a stable-to-slightly-softening plateau through
Aug-2018 -- with no evidence of a sustained multi-month decline. Proceed to
`05_export_dashboard_data.ipynb` to produce the dashboard-ready CSV.
